# 02 – Removing redundant peptides with CD-HIT (90% identity)

**Why?** Many DBAASP peptides are close relatives of each other. DBAASP itself notes that *"many of the peptides in the database are derivatives, the number of non-derivative sequences is much smaller"* and that most new peptides are artificial peptides created for structure-activity studies (Pirtskhalava et al. 2016). Different DBAASP records can also carry the same sequence with different chemical modifications. If we leave all of them in, one peptide family that was studied 100 times counts 100 times, and group comparisons (and especially amino-acid enrichment) mostly describe those big families.

**Method.** CD-HIT is a widely used program for clustering biological sequences to reduce sequence redundancy (Li & Godzik 2006; Fu et al. 2012). Sequences are sorted by length; a sequence that is at least 90% identical to an earlier one joins that cluster, otherwise it starts a new cluster. The first (longest) sequence of a cluster is the **representative**, and only representatives are kept. Filtering AMP data with CD-HIT is common practice; for example Veltri et al. (2018) removed peptides sharing high sequence identity using CD-HIT when building their AMP dataset.

**The 90% threshold** is the redundancy level chosen for this study (it is also CD-HIT's default `-c 0.9`).

**Settings and where they come from**

| Setting | Value | Source |
|---|---|---|
| `-c` identity | 0.9 | study protocol; CD-HIT default (`cd-hit -h`) |
| `-n` word size | 5 | CD-HIT user's guide: *"-n 5 for thresholds 0.7 ~ 1.0"* |
| `-l` throw-away length | 9 | CD-HIT user's guide / `cd-hit -h`: *"-l length of throw_away_sequences, default 10"*. In our tests CD-HIT drops sequences whose length is **at or below** `-l` (with `-l 5` every 5-residue sequence disappeared), so the default would lose all 10-residue peptides; `-l 9` keeps peptides of 10 residues and longer. |
| `-s` length difference | 0 (default, not set) | `cd-hit -h` |

Identity in CD-HIT is *"number of identical amino acids in alignment divided by the full length of the shorter sequence"* (`cd-hit -h`). Because `-s` is 0, a shorter peptide that matches part of a longer one is merged into the longer one (a *fragment*).

**Peptides shorter than 10 residues** are not passed to CD-HIT. For peptides of up to 9 residues, 90% identity already means *identical* (one mismatch in 9 residues is 88.9%), so we simply merge peptides that have exactly the same sequence. From 10 residues on, one mismatch is allowed (9 of 10 = 90%), so those peptides go to CD-HIT. (While testing we also saw CD-HIT leave two identical 5-residue sequences in separate clusters, another reason not to rely on it for very short peptides.)

Input: `results/clean_peptides.csv` (notebook 01). Output: `results/clean_peptides_clustered.csv` (all rows, with cluster columns) and the CD-HIT files in `results/cdhit/`.

In [1]:
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IDENTITY = 0.90            # study protocol (also CD-HIT default)
WORD_SIZE = 5              # CD-HIT user's guide: -n 5 for thresholds 0.7-1.0
MIN_LEN = 10               # from 10 residues on, 90% identity allows a mismatch (9/10); shorter peptides: identical only
CD = Path("../results/cdhit")
CD.mkdir(exist_ok=True)

df = pd.read_csv("../results/clean_peptides.csv")
print(df.shape)

(18275, 22)


## 1. Split by length and write the FASTA file

In [2]:
long_pep = df[df["length"] >= MIN_LEN]
short_pep = df[df["length"] < MIN_LEN]
print(f"{len(long_pep)} peptides with >= {MIN_LEN} residues go to CD-HIT")
print(f"{len(short_pep)} peptides with < {MIN_LEN} residues are merged by identical sequence")

with open(CD / "input.fasta", "w") as f:
    for r in long_pep.itertuples():
        f.write(f">{r.ID}\n{r.sequence}\n")

15587 peptides with >= 10 residues go to CD-HIT
2688 peptides with < 10 residues are merged by identical sequence


## 2. Run CD-HIT

In [3]:
if shutil.which("cd-hit") is None:
    raise SystemExit("cd-hit not found. Install it with `sudo apt install cd-hit` "
                     "or `conda install -c bioconda cd-hit`, then re-run this notebook.")

cmd = ["cd-hit", "-i", str(CD / "input.fasta"), "-o", str(CD / "representatives.fasta"),
       "-c", str(IDENTITY), "-n", str(WORD_SIZE),
       "-l", str(MIN_LEN - 1),                # CD-HIT drops sequences with length <= -l, so -l 9 keeps 10-residue peptides
       "-d", "0", "-M", "0", "-T", "1"]     # -d 0: keep the full ID; -M 0: no memory limit; -T 1: one thread (same result every run)
print(" ".join(cmd))
run = subprocess.run(cmd, capture_output=True, text=True, check=True)
(CD / "cdhit_run.log").write_text(run.stdout)
print(run.stdout.strip().splitlines()[-6:])

cd-hit -i ../results/cdhit/input.fasta -o ../results/cdhit/representatives.fasta -c 0.9 -n 5 -l 9 -d 0 -M 0 -T 1


['Approximated maximum memory consumption: 79M', 'writing new database', 'writing clustering information', 'program completed !', '', 'Total CPU time 0.44']


## 3. Read the clusters
Each line of the `.clstr` file is a cluster member; the member marked `*` is the representative, the others show their % identity to it.

In [4]:
members, cid = [], None
for line in open(CD / "representatives.fasta.clstr"):
    if line.startswith(">Cluster"):
        cid = f"C{int(line.split()[1])}"
        continue
    m = re.match(r"\d+\s+(\d+)aa, >(\S+?)\.\.\. (\*|at .*?(\d+\.\d+)%)", line)
    members.append({"ID": int(m.group(2)), "cluster_id": cid, "is_representative": m.group(3) == "*",
                    "identity": 100.0 if m.group(3) == "*" else float(m.group(4))})
cl = pd.DataFrame(members)

# sanity checks on the CD-HIT output
assert set(cl["ID"]) == set(long_pep["ID"]), "CD-HIT dropped or invented sequences"
assert cl["ID"].is_unique
assert (cl.groupby("cluster_id")["is_representative"].sum() == 1).all(), "each cluster needs exactly one representative"
assert cl["identity"].min() >= 100 * IDENTITY, "a member is below the identity threshold"
print(f"{len(cl)} peptides in {cl['cluster_id'].nunique()} clusters; lowest identity to a representative: {cl['identity'].min():.1f}%")

15587 peptides in 7990 clusters; lowest identity to a representative: 90.2%


## 4. Merge identical short peptides and combine

In [5]:
# short peptides: one cluster per identical sequence; representative = lowest ID
sh = short_pep[["ID", "sequence"]].sort_values("ID").copy()
sh["cluster_id"] = "S" + sh.groupby("sequence").ngroup().astype(str)
sh["is_representative"] = ~sh["sequence"].duplicated()
sh["identity"] = 100.0

clusters = pd.concat([cl, sh[["ID", "cluster_id", "is_representative", "identity"]]], ignore_index=True)
out = df.merge(clusters, on="ID", how="left", validate="one_to_one")

assert out["cluster_id"].notna().all()
assert (out.groupby("cluster_id")["is_representative"].sum() == 1).all()
out["cluster_size"] = out.groupby("cluster_id")["ID"].transform("size")
out["cluster_n_origins"] = out.groupby("cluster_id")["origin"].transform("nunique")

# the non-redundant set must not contain two identical sequences
nr = out[out["is_representative"]]
assert nr["sequence"].is_unique, "identical sequences left in the non-redundant set"
print(f"{len(out)} peptides -> {out['cluster_id'].nunique()} clusters -> {len(nr)} representatives")

18275 peptides -> 9951 clusters -> 9951 representatives


## 5. What did redundancy removal do?

In [6]:
tab = pd.DataFrame({"before": out["origin"].value_counts(), "after (representatives)": nr["origin"].value_counts()})
tab.loc["Total"] = tab.sum()
tab["kept_%"] = (100 * tab["after (representatives)"] / tab["before"]).round(1)
tab

,before,after (representatives),kept_%
origin,,,
Synthetic,15036,7650,50.9
Natural,3239,2301,71.0
Total,18275,9951,54.5


In [7]:
sizes = out.drop_duplicates("cluster_id")["cluster_size"]
print("cluster size: median", int(sizes.median()), "| clusters with 1 peptide:", int((sizes == 1).sum()),
      "| largest cluster:", int(sizes.max()))

mixed = out.drop_duplicates("cluster_id")["cluster_n_origins"].eq(2)
print(f"clusters containing BOTH natural and synthetic peptides: {mixed.sum()} of {len(sizes)} ({100*mixed.mean():.1f}%)")

# fragments: removed peptides shorter than 90% of their cluster representative
rep_len = nr.set_index("cluster_id")["length"]
non_rep = out[~out["is_representative"]]
frag = (non_rep["length"] < 0.9 * non_rep["cluster_id"].map(rep_len)).mean()
print(f"removed peptides that were shorter than 90% of their representative: {100*frag:.0f}%")
print("\nmedian length before -> after:")
print(pd.DataFrame({"before": out.groupby("origin")["length"].median(),
                    "after": nr.groupby("origin")["length"].median()}))

cluster size: median 1 | clusters with 1 peptide: 7423 | largest cluster: 102
clusters containing BOTH natural and synthetic peptides: 493 of 9951 (5.0%)
removed peptides that were shorter than 90% of their representative: 42%

median length before -> after:
           before  after
origin                  
Natural      25.0   25.0
Synthetic    15.0   14.0


In [8]:
COLORS = {"Natural": "#2a9d8f", "Synthetic": "#e76f51"}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
x = np.arange(2); w = 0.38
for k, (label, data) in enumerate([("before", out), ("after (non-redundant)", nr)]):
    counts = data["origin"].value_counts().reindex(["Natural", "Synthetic"])
    axes[0].bar(x + (k - 0.5) * w, counts.values, w, label=label, color=["#2a9d8f", "#e76f51"], alpha=1 if k else 0.4)
axes[0].set_xticks(x); axes[0].set_xticklabels(["Natural", "Synthetic"]); axes[0].set_ylabel("peptides")
axes[0].set_title("Peptides per origin", fontsize=10)
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color="grey", alpha=0.4), plt.Rectangle((0, 0), 1, 1, color="grey")],
               labels=["before", "after"], frameon=False)
bins = np.arange(0, 101, 2)
for o in ["Natural", "Synthetic"]:
    axes[1].hist(out.loc[out.origin == o, "length"].clip(upper=100), bins=bins, histtype="step", ls="--", color=COLORS[o])
    axes[1].hist(nr.loc[nr.origin == o, "length"].clip(upper=100), bins=bins, histtype="step", color=COLORS[o], label=o)
axes[1].set_xlabel("length (residues, capped at 100)"); axes[1].set_title("Length: before (dashed) vs after (solid)", fontsize=10)
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../figures/redundancy_before_after.png", dpi=150)
plt.show()

## Save

In [9]:
cols = list(df.columns) + ["cluster_id", "is_representative", "cluster_size", "cluster_n_origins"]
out[cols].to_csv("../results/clean_peptides_clustered.csv", index=False)

summary = tab.reset_index().rename(columns={"index": "origin"})
summary.to_csv("../results/redundancy_summary.csv", index=False)
print("saved", out[cols].shape)

saved (18275, 26)


## Notes for the methods / limitations section
* Redundancy was removed with CD-HIT 4.8.1 at 90% identity (`-c 0.9 -n 5 -l 9`), clustering natural and synthetic peptides **together**. Natural and synthetic peptides rarely fall in the same cluster (step 5), so this hardly moves peptides between groups.
* CD-HIT keeps the **longest** sequence of each cluster and merges shorter peptides that match part of a longer one, so the non-redundant set is biased towards longer peptides.
* Peptides shorter than 10 residues were only merged when their sequences were identical, so a short peptide that is a fragment of a longer one is *not* merged into it.
* All later notebooks use only the representatives (`is_representative == True`).

## References
* Pirtskhalava M, et al. (2016) DBAASP v.2. *Nucleic Acids Res* 44(D1):D1104-D1112. https://doi.org/10.1093/nar/gkv1174 (PMID 26578581)
* Li W, Godzik A (2006) Cd-hit: a fast program for clustering and comparing large sets of protein or nucleotide sequences. *Bioinformatics* 22(13):1658-1659. https://doi.org/10.1093/bioinformatics/btl158 (PMID 16731699)
* Fu L, Niu B, Zhu Z, Wu S, Li W (2012) CD-HIT: accelerated for clustering the next-generation sequencing data. *Bioinformatics* 28(23):3150-3152. https://doi.org/10.1093/bioinformatics/bts565 (PMID 23060610)
* Veltri D, Kamath U, Shehu A (2018) Deep learning improves antimicrobial peptide recognition. *Bioinformatics* 34(16):2740-2747. https://doi.org/10.1093/bioinformatics/bty179 (PMID 29590297)
* CD-HIT user's guide, https://github.com/weizhongli/cdhit/wiki/3.-User's-Guide (word-size table and `-l` default, checked 2026-09-29) and the `cd-hit -h` help text of version 4.8.1.